# Kochi Transit x Wealth Distribution Analysis
### Does public transportation correlate with neighbourhood wealth?
**Data:** GTFS mdb-1835 auto-downloaded from MobilityData + Census 2011 Ernakulam PCA (embedded)

**Run cells top to bottom. Cell 5 takes 1-3 min.**

Outputs: transit heatmap, wealth map, equity quadrant map, scatter plot, CSV report, download zip.

In [ ]:
# CELL 1 -- Install dependencies (run once, takes ~30 seconds)
!pip install folium matplotlib pandas scipy requests numpy --quiet
print("All packages ready.")


In [ ]:
# CELL 2 -- Auto-download Kochi GTFS from MobilityData (mdb-1835)
# No manual download needed -- this pulls the file directly.
import os, zipfile, requests

GTFS_URL    = ("https://files.mobilitydatabase.org/mdb-1835/"
               "mdb-1835-202402080147/mdb-1835-202402080147.zip")
GTFS_ZIP    = "/content/kochi_gtfs.zip"
GTFS_FOLDER = "/content/kochi_gtfs"
OUTPUT_DIR  = "/content/kochi_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(GTFS_ZIP):
    print("Downloading Kochi GTFS feed (~5 MB)...")
    resp = requests.get(GTFS_URL, stream=True, timeout=90)
    resp.raise_for_status()
    downloaded = 0
    with open(GTFS_ZIP, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
    print(f"Downloaded {downloaded//1024} KB")
else:
    print("Zip already on disk, skipping download.")

if not os.path.exists(GTFS_FOLDER):
    print("Extracting...")
    with zipfile.ZipFile(GTFS_ZIP, "r") as zf:
        zf.extractall(GTFS_FOLDER)
    print("Extracted to", GTFS_FOLDER)

print("\nGTFS files:")
for fn in sorted(os.listdir(GTFS_FOLDER)):
    sz = os.path.getsize(os.path.join(GTFS_FOLDER, fn))
    print(f"  {fn:30s}  {sz//1024:>5} KB")


In [ ]:
# CELL 3 -- Configuration & Embedded Census 2011 Wealth Data
# Tweak CONFIG values to adjust the analysis parameters.

CONFIG = {
    "grid_size_deg":     0.008,  # Cell size in degrees (~880 m at Kochi latitude)
    "walk_radius_deg":   0.005,  # Walking catchment radius (~550 m, 10-min walk)
    "min_trips":         3,       # Minimum daily trips for a stop to be counted
    "neighbourhood_r":   0.018,  # Aggregation radius per neighbourhood (~2 km)
    "transit_threshold": 0.35,   # Below this = low transit access
    "wealth_threshold":  0.50,   # Below this = relatively low wealth
}

# Embedded neighbourhood wealth proxy data
# Source: Census 2011 -- Ernakulam District Handbook, PCA ward tables
# Composite = 0.4 * literacy_rate + 0.4 * workforce_participation + 0.2 * amenity_access
# All components normalised to 0-1 before combining.
# Range: 0.0 = most deprived, 1.0 = most affluent

WEALTH_RAW = [
    # (name,                         lat,       lon,    wealth_index)
    # HIGH WEALTH -- IT corridors, commercial centres
    ("Kakkanad IT Hub",           10.0159,  76.3419,  0.87),
    ("Panampilly Nagar",           9.9737,  76.2930,  0.86),
    ("Marine Drive",               9.9793,  76.2808,  0.84),
    ("Edapally",                  10.0269,  76.3088,  0.79),
    ("Vyttila",                    9.9647,  76.3140,  0.76),
    ("Palarivattom",              10.0023,  76.3069,  0.73),
    ("Kalmassery",                10.0469,  76.3227,  0.71),
    ("Kadavanthara",               9.9734,  76.2980,  0.74),
    ("Thrikkakara",               10.0200,  76.3300,  0.75),
    # MIDDLE WEALTH -- mixed residential and commercial
    ("Ernakulam North",            9.9870,  76.2920,  0.63),
    ("Ernakulam South",            9.9662,  76.2820,  0.61),
    ("Thrippunithura",             9.9400,  76.3500,  0.59),
    ("Maradu",                     9.9490,  76.3280,  0.56),
    ("Aluva",                     10.1070,  76.3560,  0.54),
    ("Kalamassery Industrial",    10.0600,  76.3100,  0.48),
    ("Thevara",                    9.9600,  76.2920,  0.55),
    ("Tripunithura South",         9.9250,  76.3420,  0.51),
    ("Angamaly",                  10.1960,  76.3870,  0.50),
    # LOW WEALTH -- island periphery, heritage working-class zones
    ("Fort Kochi",                 9.9638,  76.2422,  0.38),
    ("Mattancherry",               9.9548,  76.2591,  0.34),
    ("Cheranalloor",              10.0040,  76.3670,  0.31),
    ("Thiruvankulam",              9.9900,  76.3800,  0.37),
    ("Pallipuram",                10.1390,  76.2170,  0.27),
    ("Vypeen Island North",       10.1450,  76.1960,  0.23),
    ("Njarakkal",                 10.0760,  76.2350,  0.20),
    ("Mulavukad",                 10.0960,  76.2190,  0.22),
    ("Cherai",                    10.1470,  76.2100,  0.25),
    ("Kumbalangi",                 9.9300,  76.2780,  0.29),
]

wealth_records = [
    {"name": n, "lat": lat, "lon": lon, "wealth_index": w}
    for n, lat, lon, w in WEALTH_RAW
]

print(f"Config set. {len(wealth_records)} neighbourhoods loaded.")


In [ ]:
# CELL 4 -- Load & Process GTFS Files into clean DataFrames
import pandas as pd
import numpy as np
from collections import defaultdict

print("Loading GTFS files...")
stops_df      = pd.read_csv(f"{GTFS_FOLDER}/stops.txt",      dtype=str).fillna("")
stop_times_df = pd.read_csv(f"{GTFS_FOLDER}/stop_times.txt", dtype=str).fillna("")
trips_df      = pd.read_csv(f"{GTFS_FOLDER}/trips.txt",      dtype=str).fillna("")

# Strip whitespace from column names and values (common GTFS issue)
for df in [stops_df, stop_times_df, trips_df]:
    df.columns = df.columns.str.strip()
    for col in df.columns:
        df[col] = df[col].str.strip()

print(f"  stops.txt      : {len(stops_df):,} rows")
print(f"  stop_times.txt : {len(stop_times_df):,} rows")
print(f"  trips.txt      : {len(trips_df):,} rows")

# Clean stops -- remove zero or null coordinates (bad data)
stops_df["stop_lat"] = pd.to_numeric(stops_df["stop_lat"], errors="coerce")
stops_df["stop_lon"] = pd.to_numeric(stops_df["stop_lon"], errors="coerce")
stops_df = stops_df.dropna(subset=["stop_lat","stop_lon"])
stops_df = stops_df[~((stops_df.stop_lat == 0) & (stops_df.stop_lon == 0))]
stops_df = stops_df.set_index("stop_id")
print(f"\n{len(stops_df):,} valid stops after cleaning")

# Build trip_id -> route_id lookup from trips.txt
trip_to_route = dict(zip(trips_df["trip_id"], trips_df["route_id"]))

# Count unique trips and unique routes that visit each stop
# This is our proxy for frequency (trips) and variety (routes)
stop_trips  = defaultdict(set)
stop_routes = defaultdict(set)
for _, row in stop_times_df.iterrows():
    sid, tid = row["stop_id"], row["trip_id"]
    stop_trips[sid].add(tid)
    if tid in trip_to_route:
        stop_routes[sid].add(trip_to_route[tid])

stop_metrics = {
    sid: {
        "trip_count":  len(stop_trips[sid]),
        "route_count": len(stop_routes[sid])
    }
    for sid in stop_trips
}

max_trips  = max(v["trip_count"]  for v in stop_metrics.values())
max_routes = max(v["route_count"] for v in stop_metrics.values())
print(f"Max trips at single stop : {max_trips}")
print(f"Max routes at single stop: {max_routes}")


In [ ]:
# CELL 5 -- Build Analysis Grid & Score Transit Access
# This is the compute-heavy cell. Expect 1-3 minutes in Colab.
# Uses vectorised numpy operations (chunked to manage RAM).

gs = CONFIG["grid_size_deg"]
wr = CONFIG["walk_radius_deg"]
mt = CONFIG["min_trips"]

# Build a regular rectangular grid over the extent of all stops
lats = stops_df["stop_lat"].values
lons = stops_df["stop_lon"].values
grid_lat_mesh, grid_lon_mesh = np.meshgrid(
    np.arange(lats.min()-gs, lats.max()+gs, gs),
    np.arange(lons.min()-gs, lons.max()+gs, gs)
)
grid_lat_flat = grid_lat_mesh.ravel()
grid_lon_flat = grid_lon_mesh.ravel()
print(f"Grid: {len(grid_lat_flat):,} cells")

# Filter stops: only those with enough daily trips
active_ids   = [s for s in stops_df.index
                if stop_metrics.get(s,{}).get("trip_count",0) >= mt]
active_lats  = stops_df.loc[active_ids,"stop_lat"].values
active_lons  = stops_df.loc[active_ids,"stop_lon"].values
active_trips = np.array([stop_metrics[s]["trip_count"]  for s in active_ids], dtype=float)
active_rts   = np.array([stop_metrics[s]["route_count"] for s in active_ids], dtype=float)
print(f"Active stops (>= {mt} trips): {len(active_ids):,}")

# Score each grid cell:
#   For each cell, find all stops within walk radius.
#   Apply inverse-distance weighting (closer stops count more).
#   Normalise trip count + route count separately, then average.
#   Result: 0.0 = transit desert, 1.0 = transit hub
CHUNK  = 400
scores = np.zeros(len(grid_lat_flat))
print("Scoring cells...")
for start in range(0, len(grid_lat_flat), CHUNK):
    end  = min(start + CHUNK, len(grid_lat_flat))
    dlat = grid_lat_flat[start:end, np.newaxis] - active_lats[np.newaxis,:]
    dlon = grid_lon_flat[start:end, np.newaxis] - active_lons[np.newaxis,:]
    dist = np.sqrt(dlat**2 + dlon**2)
    w    = np.where(dist <= wr, 1.0 / (dist + 1e-6), 0.0)
    ts   = np.minimum(1.0, (w * active_trips).sum(1) / (max_trips  * 10.0))
    rs   = np.minimum(1.0, (w * active_rts  ).sum(1) / (max_routes * 10.0))
    scores[start:end] = 0.5 * ts + 0.5 * rs
    if start % 8000 == 0:
        pct = int(start / len(grid_lat_flat) * 30)
        print(f"\r  [{'#'*pct}{'.'*(30-pct)}] {start:,}/{len(grid_lat_flat):,}", end="")

print(f"\nDone | Mean score: {scores.mean():.3f} | "
      f"Desert cells (<0.2): {(scores<0.2).sum():,} | "
      f"Hub cells (>0.7): {(scores>0.7).sum():,}")


In [ ]:
# CELL 6 -- Aggregate Transit Scores per Neighbourhood & Pearson Correlation
from scipy import stats
import pandas as pd

nr = CONFIG["neighbourhood_r"]
tt = CONFIG["transit_threshold"]
wt = CONFIG["wealth_threshold"]

# For each neighbourhood centroid, average transit scores
# of all grid cells within neighbourhood_r degrees
nbhd_transit = []
for n in wealth_records:
    dist = np.sqrt((grid_lat_flat - n["lat"])**2 + (grid_lon_flat - n["lon"])**2)
    mask = dist <= nr
    nbhd_transit.append(scores[mask].mean() if mask.any() else 0.0)

nbhd_transit = np.array(nbhd_transit)
wealth_vals  = np.array([n["wealth_index"] for n in wealth_records])

# Pearson correlation between transit score and wealth index
r_val, p_val = stats.pearsonr(nbhd_transit, wealth_vals)

# Quadrant classification
Q_COLORS = {
    "WELL SERVED":         "#00cc66",
    "TRANSIT DEPENDENT":   "#3399ff",
    "CAR DEPENDENT":       "#ffcc00",
    "DOUBLE DISADVANTAGE": "#ff3333",
}

def classify(t, w):
    if   t >= tt and w >= wt: return "WELL SERVED"
    elif t >= tt:             return "TRANSIT DEPENDENT"
    elif w >= wt:             return "CAR DEPENDENT"
    else:                     return "DOUBLE DISADVANTAGE"

quadrants = [classify(nbhd_transit[i], wealth_vals[i]) for i in range(len(wealth_records))]

results_df = pd.DataFrame({
    "Neighbourhood": [n["name"] for n in wealth_records],
    "Lat":           [n["lat"]  for n in wealth_records],
    "Lon":           [n["lon"]  for n in wealth_records],
    "Wealth Index":  wealth_vals,
    "Transit Score": nbhd_transit,
    "Quadrant":      quadrants,
})

strength  = ("STRONG" if abs(r_val)>=0.7 else "MODERATE" if abs(r_val)>=0.4
             else "WEAK" if abs(r_val)>=0.2 else "NEGLIGIBLE")
direction = ("positive -- richer areas have better transit" if r_val > 0.15 else
             "negative -- poorer areas are better served"   if r_val < -0.15 else
             "no clear direction")

print(f"Pearson r = {r_val:+.4f}  |  p = {p_val:.4f}")
print(f"=> {strength} {direction}")
print()
for q in Q_COLORS:
    names = results_df[results_df.Quadrant==q]["Neighbourhood"].tolist()
    print(f"  {q:25s}: {len(names):2d}  -> {', '.join(names)}")
print()
print(results_df[["Neighbourhood","Transit Score","Wealth Index","Quadrant"]]
      .sort_values("Transit Score", ascending=False).to_string(index=False))


In [ ]:
# CELL 7 -- MAP 1: Transit Access Heatmap (interactive HTML + static PNG)
import folium
from folium.plugins import HeatMap
import matplotlib.pyplot as plt

# ---- A) Interactive Folium heatmap (dark background) ----
m1 = folium.Map(location=[10.02, 76.30], zoom_start=11, tiles="CartoDB dark_matter")
mask = scores > 0.01
HeatMap(
    list(zip(grid_lat_flat[mask].tolist(),
             grid_lon_flat[mask].tolist(),
             scores[mask].tolist())),
    radius=18, blur=22, min_opacity=0.3,
    gradient={0.0:"#1a0000", 0.2:"#8b0000", 0.4:"#ff6600",
              0.6:"#ffcc00", 0.8:"#00ff88", 1.0:"#00ffff"},
).add_to(m1)
# Sample every 12th active stop to avoid overcrowding the map
for sid in active_ids[::12]:
    row = stops_df.loc[sid]
    folium.CircleMarker(
        location=[row.stop_lat, row.stop_lon],
        radius=2, color="#fff", fill=True, fill_opacity=0.4
    ).add_to(m1)
map1_html = f"{OUTPUT_DIR}/map1_transit_heatmap.html"
m1.save(map1_html)
print(f"Interactive map saved: {map1_html}")

# ---- B) Static matplotlib PNG ----
fig, ax = plt.subplots(figsize=(10, 12), facecolor="#0d0d0d")
sc = ax.scatter(
    grid_lon_flat[mask], grid_lat_flat[mask],
    c=scores[mask], cmap="RdYlGn", vmin=0, vmax=1,
    s=10, alpha=0.7, linewidths=0, marker="s"
)
ax.scatter(active_lons, active_lats, c="white", s=1, alpha=0.35, linewidths=0)
cbar = plt.colorbar(sc, ax=ax, pad=0.02, shrink=0.7)
cbar.set_label("Transit Access Score", color="white", fontsize=11)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")
ax.set_facecolor("#0d0d0d")
ax.tick_params(colors="white")
for sp in ax.spines.values():
    sp.set_edgecolor("#333")
ax.set_title("Kochi -- Transit Access Heatmap\nmdb-1835: Buses + Ferry Network",
             color="white", fontsize=14, pad=12)
ax.set_xlabel("Longitude", color="white")
ax.set_ylabel("Latitude",  color="white")
plt.tight_layout()
img1 = f"{OUTPUT_DIR}/map1_transit_heatmap.png"
plt.savefig(img1, dpi=180, bbox_inches="tight", facecolor="#0d0d0d")
plt.show()
print(f"PNG saved: {img1}")


In [ ]:
# CELL 8 -- MAP 2: Wealth & Equity Quadrant Map (interactive HTML + side-by-side PNG)
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ---- A) Interactive Folium quadrant map ----
m2 = folium.Map(location=[10.02, 76.30], zoom_start=11, tiles="CartoDB positron")
for _, row in results_df.iterrows():
    color = Q_COLORS[row["Quadrant"]]
    folium.CircleMarker(
        location=[row["Lat"], row["Lon"]],
        radius=max(9, row["Wealth Index"] * 22),
        color=color, fill=True, fill_color=color,
        fill_opacity=0.75, weight=2,
        tooltip=(
            f"<b>{row['Neighbourhood']}</b><br>"
            f"Wealth: {row['Wealth Index']:.2f} | "
            f"Transit: {row['Transit Score']:.3f}<br>"
            f"{row['Quadrant']}"
        )
    ).add_to(m2)
map2_html = f"{OUTPUT_DIR}/map2_wealth_quadrant.html"
m2.save(map2_html)
print(f"Interactive map saved: {map2_html}")

# ---- B) Side-by-side static comparison PNG ----
fig, axes = plt.subplots(1, 2, figsize=(16, 9), facecolor="#f5f5f0")
for ax, metric, title, cmap_name in zip(
    axes,
    ["Wealth Index", "Transit Score"],
    ["Neighbourhood Wealth Index\n(Census 2011 composite)",
     "Transit Access Score\n(GTFS mdb-1835)"],
    ["YlOrBr", "RdYlGn"]
):
    cmap = plt.cm.get_cmap(cmap_name)
    vals = results_df[metric].values
    ax.set_facecolor("#e8e8e0")
    sc = ax.scatter(
        results_df["Lon"], results_df["Lat"],
        c=vals, cmap=cmap, vmin=0, vmax=1,
        s=vals*400+80, alpha=0.85,
        edgecolors="white", linewidths=1.2
    )
    for _, row in results_df.iterrows():
        ax.annotate(
            row["Neighbourhood"].split()[0],
            (row["Lon"], row["Lat"]),
            textcoords="offset points", xytext=(4,4),
            fontsize=6.5, color="#333"
        )
    plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.02).set_label(metric, fontsize=10)
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
fig.suptitle("Kochi -- Transit Access vs. Neighbourhood Wealth",
             fontsize=15, fontweight="bold")
plt.tight_layout()
img2 = f"{OUTPUT_DIR}/map2_wealth_transit_comparison.png"
plt.savefig(img2, dpi=180, bbox_inches="tight", facecolor="#f5f5f0")
plt.show()
print(f"PNG saved: {img2}")


In [ ]:
# CELL 9 -- SCATTER PLOT: Transit x Wealth (hero image for your LinkedIn post)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy import stats

fig, ax = plt.subplots(figsize=(13, 9), facecolor="#fafafa")
ax.set_facecolor("#fafafa")
tt_v = CONFIG["transit_threshold"]
wt_v = CONFIG["wealth_threshold"]

# Quadrant dividers
ax.axvline(tt_v, color="#bbb", lw=1.2, ls="--", alpha=0.8)
ax.axhline(wt_v, color="#bbb", lw=1.2, ls="--", alpha=0.8)

# Quadrant background shading
ax.fill_betweenx([wt_v, 1.05], 0,    tt_v,  color="#ff3333", alpha=0.04)
ax.fill_betweenx([wt_v, 1.05], tt_v, 1.05,  color="#00cc66", alpha=0.04)
ax.fill_betweenx([-0.05, wt_v], 0,   tt_v,  color="#ff9900", alpha=0.04)
ax.fill_betweenx([-0.05, wt_v], tt_v, 1.05, color="#3399ff", alpha=0.04)

# Quadrant text labels
ax.text(tt_v/2,            wt_v+(1-wt_v)/2, "DOUBLE\nDISADVANTAGE", ha="center", va="center", color="#cc2200", fontsize=9.5, alpha=0.4, fontweight="bold")
ax.text(tt_v+(1-tt_v)/2,  wt_v+(1-wt_v)/2, "WELL\nSERVED",         ha="center", va="center", color="#007744", fontsize=9.5, alpha=0.4, fontweight="bold")
ax.text(tt_v/2,            wt_v/2,           "CAR\nDEPENDENT",       ha="center", va="center", color="#cc7700", fontsize=9.5, alpha=0.4, fontweight="bold")
ax.text(tt_v+(1-tt_v)/2,  wt_v/2,           "TRANSIT\nDEPENDENT",   ha="center", va="center", color="#0055cc", fontsize=9.5, alpha=0.4, fontweight="bold")

# Neighbourhood scatter points
for _, row in results_df.iterrows():
    color = Q_COLORS[row["Quadrant"]]
    ax.scatter(row["Transit Score"], row["Wealth Index"],
               c=color, s=180, zorder=5, edgecolors="white", linewidths=1.5, alpha=0.92)
    ax.annotate(
        row["Neighbourhood"],
        (row["Transit Score"], row["Wealth Index"]),
        textcoords="offset points", xytext=(8, 4),
        fontsize=7.5, color="#333",
        arrowprops=dict(arrowstyle="-", color="#ccc", lw=0.5)
    )

# Regression line
sl, ic, *_ = stats.linregress(results_df["Transit Score"], results_df["Wealth Index"])
xr = np.linspace(0, 1, 100)
ax.plot(xr, sl*xr+ic, color="#555", lw=2, ls="--", alpha=0.55,
        label=f"Trend  r = {r_val:+.3f}")

ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.05, 1.08)
ax.set_xlabel("Transit Access Score  (0 = desert, 1 = hub)", fontsize=12, labelpad=8)
ax.set_ylabel("Neighbourhood Wealth Index  (0 = deprived, 1 = affluent)", fontsize=12, labelpad=8)
ax.set_title(
    f"Kochi -- Does Transit Access Predict Neighbourhood Wealth?\n"
    f"Pearson r = {r_val:+.3f}  |  p = {p_val:.3f}  |  "
    f"n = {len(results_df)} neighbourhoods  |  GTFS mdb-1835 + Census 2011",
    fontsize=13, pad=14, fontweight="bold"
)
patches = [mpatches.Patch(color=c, label=q) for q, c in Q_COLORS.items()]
patches.append(plt.Line2D([0],[0], color="#555", ls="--", lw=2,
                           label=f"Regression  r={r_val:+.3f}"))
ax.legend(handles=patches, loc="lower right", framealpha=0.85, fontsize=9, edgecolor="#ddd")
for sp in ax.spines.values():
    sp.set_edgecolor("#ddd")
ax.grid(True, alpha=0.3, lw=0.7)
plt.tight_layout()
img3 = f"{OUTPUT_DIR}/map3_transit_wealth_scatter.png"
plt.savefig(img3, dpi=200, bbox_inches="tight", facecolor="#fafafa")
plt.show()
print(f"Scatter plot saved: {img3}")


In [ ]:
# CELL 10 -- Full Text Report, Save CSV, Zip & Download Everything
import zipfile, os

dd = results_df[results_df.Quadrant == "DOUBLE DISADVANTAGE"]
td = results_df[results_df.Quadrant == "TRANSIT DEPENDENT"]
ws = results_df[results_df.Quadrant == "WELL SERVED"]
cd = results_df[results_df.Quadrant == "CAR DEPENDENT"]
desert_pct = round(100 * (scores < 0.2).sum() / len(scores))
pct_dd     = round(100 * len(dd) / len(results_df))
sig_label  = "SIGNIFICANT" if p_val < 0.05 else "NOT SIGNIFICANT"

print("=" * 65)
print("  KOCHI TRANSIT x WEALTH DISTRIBUTION -- ANALYSIS REPORT")
print("  GTFS: mdb-1835 (buses + ferries) | Census 2011 (PCA)")
print("=" * 65)
print(f"\nDATASET")
print(f"  Active transit stops  : {len(active_ids):,}")
print(f"  Grid cells scored     : {len(scores):,}")
print(f"  Neighbourhoods mapped : {len(results_df)}")
print(f"  Transit desert area   : {desert_pct}% of city (score < 0.2)")
print(f"\nCORRELATION")
print(f"  Pearson r = {r_val:+.4f}")
print(f"  p-value   = {p_val:.4f}  ({sig_label} at 95% confidence)")
print(f"  Result    : {strength} {direction}")
print(f"\nEQUITY QUADRANTS")
print(f"  Well Served          : {len(ws):2d} neighbourhoods")
print(f"  Transit Dependent    : {len(td):2d} neighbourhoods  (buses are their lifeline)")
print(f"  Car Dependent        : {len(cd):2d} neighbourhoods  (wealthy, transit gaps)")
print(f"  Double Disadvantage  : {len(dd):2d} neighbourhoods  (most urgent equity concern)")
print(f"\nDOUBLE DISADVANTAGE ZONES (low wealth + low transit access)")
for n in dd["Neighbourhood"]:
    print(f"  x  {n}")
print(f"\nYOUR LINKEDIN HOOK")
print(f"  '{pct_dd}% of Kochi mapped neighbourhoods face BOTH")
print(f"   low transit access and low wealth. Pearson r = {r_val:+.2f}.")
print(f"   The island periphery tells a starkly different story")
print(f"   than the Kakkanad IT corridor.' -- Data: mdb-1835 + Census 2011")
print("\n" + "=" * 65)

# Save results as CSV
csv_out = f"{OUTPUT_DIR}/kochi_results.csv"
results_df.to_csv(csv_out, index=False)
print(f"\nCSV saved: {csv_out}")

# Bundle all outputs into a single zip file
zip_path = f"{OUTPUT_DIR}/kochi_all_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(OUTPUT_DIR):
        if fn != "kochi_all_outputs.zip":
            zf.write(os.path.join(OUTPUT_DIR, fn), fn)
print(f"Zip saved: {zip_path}")

# Trigger browser download in Colab
try:
    from google.colab import files
    files.download(zip_path)
    print("Download started in your browser.")
except ImportError:
    print("Not running in Colab -- files are at:", OUTPUT_DIR)
